# Customer Loan Dataset and Multi-Machine Learning Model Evaluation

This project uses the Loan Dataset by Kaggle user prakashraushan, available at https://www.kaggle.com/datasets/prakashraushan/loan-dataset

Important information about dataset, as per link:

"This dataset contains information about customer loans, including customer demographics, loan details, and default status. The dataset can be used for various data analysis and machine learning tasks, such as predicting loan default risk. The dataset consists of the following columns:

customer_id: Unique identifier for each customer

customer_age: Age of the customer

customer_income: Annual income of the customer

home_ownership: Home ownership status (e.g., RENT, OWN, MORTGAGE)

employment_duration: Duration of employment in months

loan_intent: Purpose of the loan (e.g., PERSONAL, EDUCATION, MEDICAL, VENTURE)

loan_grade: Grade assigned to the loan

loan_amnt: Loan amount requested

loan_int_rate: Interest rate of the loan

term_years: Loan term in years

historical_default: Indicates if the customer has a history of default (Y/N)

cred_hist_length: Length of the customer's credit history in years

Current_loan_status: Current status of the loan (DEFAULT, NO DEFAULT)"

# 1. Important Terms

1. Accuracy - overall correctness of the model. TP + TN / TP+TN+FP+FN\
    best used when classes are balanced 
2. Precision - accuracy of positive predictions ie. "out of all instances the model predicted as positive, how many were actually correct?"
    TP / TP + FP    
    Best used when you want to minimize False Positives. 

3. Recall - the ability of model to find all positive instances. ie. "out of all actual positives, how many did the model find?"
    TP / TP + FN
    Best used to minimize False Negatives.
    Best used in the loan default scenario

4. F1-score - the harmonic mean of precision and recall. 
    Especially useful with uneven call distribution

**WHATS BEST FOR THIS ANALYSIS, PRECISION OR RECALL?**
If I think about the context of this being a loan default risk dataset, I 
can identify the consequences of FPs and the consequences of FNs.

**Positive = DEFAULT (1)**

**Negative = NO DEFAULT (0)**

If a company approves a loan for a customer who will default (false negative),
the company will lose a lot of money

On the other hand, if the company disproves as loan for a customer who won't default (false positive), the company misses out on a customer, but doesn't lose money.

In reality, a loan company would need to balance this with what the real-world financial data says about the costs of false positives vs. false negatives. However, taking this at face value, false positives are not nearly as damaging as false negatives. So for this dataset 
we need to minimize false negatives, which is what recall does.

I should also think about F1 score which strikes a healthy balance between precision and recall, and also accuracy.

IN OTHER WORDS...

Precision - "When I predict DEFAULT, how often am I right?"

Recall - "Out of everyone who actually defaulted, how many did I catch?"

In [ ]:
import sys
!{sys.executable} -m pip install statsmodels

Import all required libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import (
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix, 
    precision_score, 
    recall_score, 
    f1_score,
    accuracy_score,
    PrecisionRecallDisplay
)

from sklearn.model_selection import (
    train_test_split,
    cross_val_predict,
    GridSearchCV
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


# 2. Reading and Cleaning Data

In [ ]:
df = pd.read_csv('data/loan_dataset.csv')
print( df.shape )

Some extra details on dataset:

1. TARGET VALUE is Current Loan Status. Binary: Default, and no Default. Default = True and No Default = False. Use other attributes to predict loan status.

2. Different data types to be handled. (nominal, ordinal, ratio, etc).

3. Contains missing values.

4. Has a lot of other quirks to it. Data definitely needs to be cleaned before use.

    Example: Loan amount column has '£' and ',' symbols, so instead of being numerical it is a string. That needs to be removed


In [ ]:
#looking at datatypes for each column
for x in df.columns:
  print( str( type( df[x].iloc[0] ) ) + ": " + x )

In [ ]:
df.count()

Some of these columns having missing values.

    employment_duration, loan_amnt, loan_int_rate, historical_default, and current_loan_status (target)

I've already chosen to replace null values with mean for employment duration and loan int rate

Historical default has an extremely large number of missing values because it's only applicable to customers who's current loan status is DEFAULT. Instead of making it binary I'm just gonna make it 3 options: the null value will elicit a third option (either 0 or 2).



# 2.1 Convert Categorical to Numerical
I need to convert categories to numeric values so I can run models.
1. Target variable (y) Current_loan_status

DEFAULT = 1

NO DEFAULT = 0

In [ ]:
# Converting Default and No Default (Target variable y) to 1 and 0 respectively, and filling Nan values.
df['Current_loan_status'] = df['Current_loan_status'].replace('DEFAULT', 1)
df['Current_loan_status'] = df['Current_loan_status'].replace('NO DEFAULT', 0)
df['Current_loan_status'] = df['Current_loan_status'].fillna(0)

OneHotEncoder: for independent feature variables (X)

LabelEncoder: for dependent target variable (y)

2. home_ownership
3. loan_intent

In [ ]:
categories = ['home_ownership', 'loan_intent']

encoder = OneHotEncoder(sparse_output=False, handle_unknown ='ignore')

encoded_features = encoder.fit_transform(df[categories].astype(str))

encoded_df = pd.DataFrame(encoded_features,
            columns=encoder.get_feature_names_out(categories),
            index=df.index
            )

df = pd.concat([df.drop(columns=categories), encoded_df],
               axis = 1
               )

print(df.head())

4. Handle historical_default and its missing values

In [ ]:
print(df['historical_default'].value_counts(dropna=False))

In [ ]:
df['historical_default'] = df['historical_default'].fillna('N')

df['historical_default'] = df['historical_default'].map({
    'N': 0,
    'Y': 1
}).astype(int)

In [ ]:
df['Current_loan_status'] = df['Current_loan_status'].astype(int)

print(df['Current_loan_status'].dtype)
print(df['Current_loan_status'].unique())

For Loan_grade column, I want to preserve its ordinal nature so use OrdinalEncoder


In [ ]:
encoder = OrdinalEncoder()
df['loan_grade'] = encoder.fit_transform(df[['loan_grade']])
print(df['loan_grade'])

# A = 0, b = 1, etc.


Checking now the types of our variables, they either need to be int64 or float64

In [ ]:
print(df.dtypes)

customer_income and loan_amt are both str, customer_income needs to be int and loan_amt needs to be float

In [ ]:
# Now to remove the '£' and ',' symbols from the loan amount column

df['loan_amnt'] = df['loan_amnt'].str.replace('£', '')
df['loan_amnt'] = df['loan_amnt'].str.replace(',', '')
df['loan_amnt'] = df['loan_amnt'].astype(float)

# I also had to do this with customer_income because there was ONE row that had a comma in it.
# That took me a long time to notice

df['customer_income'] = df['customer_income'].str.replace(',', '')
df['customer_income'] = df['customer_income'].astype(int)

df['loan_amnt']

# 2.2 Handling Missing Values


Need to fill in Employment_Duration, loan_amnt, loan_int_rate,

The link where I got the dataset from didn't specify why there were missing values for certain columns, so I'm gonna have to make some (reasonable) assumptions and work from there.

FOR EMPLOYMENT DURATION: I was thinking that null value meant 0, but there are actual 0 values in the column, so that can't be it. Therefore, I'm gonna assume that that data was simply unavailable, and replace the null values with the mean value

FOR LOAN AMOUNT: The loan amount is the loan amount *requested by customer, not the approved loan amount. I'm gonna assume that the missing values represent unspecified loan amount values for whatever reason
and replace them with the mean values.

FOR LOAN INT RATE: There are no 0 values, so the loans with a null loan_int_rate are no-interest loans
which do exist. So these null values can be replaced with 0.

FOR CURRENT LOAN STATUS: I noticed that there were 4 missing values. I handled this issue earlier when setting the dataframe to binary 0 and 1, and I simply filled in the null values with No Default, because that is the most common outcome (79%).

In [ ]:
df['employment_duration'] = df['employment_duration'].fillna(df['employment_duration'].mean())
df['loan_amnt'] = df['loan_amnt'].fillna(df['loan_amnt'].mean())
df['loan_int_rate'] = df['loan_int_rate'].fillna(0)

df.count()


As a side note, Customer_id has 3 missing values, which is strange. I'm not sure why that's the case, but it's not included in the dataframe because it's not gonna be relevant for the models anyway.

# 3. Exploratory Analysis

I want to see how the variables interact with each other, and which variables matter. I want to do some feature selection based off this, omitting features, potentially combining features.


In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
#double-checking to see if all data is numerical now
df.info()

In [ ]:
#Checking for correlations
correlation_matrix = df.corr()

print(np.round(correlation_matrix, 4))

In [ ]:
# focusing on target variable
correlation_matrix['Current_loan_status'].sort_values(ascending = False)

loan_grade, home_ownership_rent, loan_int_rate, and historical_default have stronger positive correlation with Current_loan_status. A higher Current_loan_status correlation represents **default** (=1)

home_ownership_OWN, customer_income, and home_ownership_MORTGAGE all correlate negatively, or with **no default** (=0)

loan_amnt, home_ownership_OTHER, cred_hist_length, and loan_intent_PERSONAL appear to have very little correlation

# 4. K Nearest Neighbors


This is clearly a classification problem. I want to run KNN, Decision Trees, and Random Forest on it.

*Credits: I used some code from previous programming assignments to help me with these models, and I made adjustments to them.

Sklearn also helped me implement these models, particularly the pages for KNN and Decision Trees.
https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

Splitting data into test and train


**Dropping Features:** loan_amnt, home_ownership_OTHER, cred_hist_length and loan_intent_PERSONAL all have very little correlation with target variable so by dropping them one at a time, I may reduce the noise in my models, improving them, but I have to be careful.

I also want to try historical_default because for that one I had to replace a lot of missing values

In [ ]:
X_knn = df.drop(['home_ownership_OTHER', 'customer_id', 'historical_default', 'Current_loan_status'], axis = 1)
y = df['Current_loan_status'] 
X_train_knn, X_test_knn, y_train, y_test = train_test_split(
    X_knn, 
    y, 
    random_state = 42,
    test_size = 0.2, 
    shuffle = True)

In the end I dropped home_ownership_OTHER and historical_default improved knn, will have to test if this works for future models as well

Scaler

In [ ]:
scaler = RobustScaler()
X_train_knn = scaler.fit_transform(X_train_knn)
X_test_knn = scaler.transform(X_test_knn)

In [ ]:
K = []
training = []
test = []
scores = {}

best_recall = 0
best_k = 1

For my KNN model, I did the following
- dropped historical_default, customer_id, and home_ownership_OTHER
- scaled with StandardScaler (above)
- Searched for and used the best k value that gave the best recall
- used manhattan metric for standard euclidean as it improves recall
    - manhattan distance is better suuited for datasets with many variables/features, such as this one

In [ ]:
results = [] 

thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, .24, .23, .22, .21, .20, .15, .10, .05]

# test each k value using CROSS VALIDATION on the training data only.
# testing each k value.
#the loop starts at k=3 because 1 and 2 are too small, and
# it goes up to 32 in increments of 2 to avoid ties in voting.
for k in range(3, 32, 2): 
    knn = KNeighborsClassifier(
        n_neighbors = k, 
        # distance weighting yields better results over uniform weighting
        weights = 'distance', 
        # manhattan yields better results over euclidean  
        metric='manhattan', 
        )

    #Generating cross-validated predictions for the training data using the current k value
    probabilities = cross_val_predict(
        knn, 
        X_train_knn, 
        y_train, 
        cv=5, 
        method='predict_proba',
        n_jobs=-1
        )[:, 1]
    
    # I try multiple thresholds to see which gives me
    # the best results specifically in terms of recall.
    # sklearn defaults to a threshold of .5
    # meaing probability _> .5 = 1 (DEFAULT), probability < .5 = 0 (NO DEFAULT)
    # but if I lower it to .3, then probability _> .3 = 1, probability < .3 = 0
    # so basically by lowering threshold the model becomes less strict (more people are considered defaulters)
    # and reduces the number of false negatives(higher recall), which is what I want to do in this case.
    for threshold in thresholds: 
        y_pred_cv = (probabilities >= threshold).astype(int)

        precision = precision_score(
            y_train, y_pred_cv, zero_division = 0
            )
        
        recall = recall_score(y_train, y_pred_cv)
        f1 = f1_score(y_train, y_pred_cv)

        results.append({
            'k': k,
            'threshold': threshold,
            'precision': precision,
            'recall': recall,
            'f1': f1
        })

# I don't want precision to be horrible, so I filter it
# to only include results where precision is at least 0.50.
acceptable_results = [ 
    result 
    for result in results
    if result["precision"] >= 0.50
]

if not acceptable_results:
    raise ValueError(
        "No tested threshold produced precision of at least 0.50."
    )

best_result = max(  # find the best recall out of the acceptable results
    acceptable_results,
    key=lambda result: result["recall"]
)

best_k = best_result["k"]
best_threshold = best_result["threshold"]
print("Best CV result:", best_result)


Final KNN model

In [ ]:
# now that k and threshold have been selected without test set,
# fit the final KNN using all training data

knn = KNeighborsClassifier(  # run knn again with best k 
    n_neighbors=best_k,
    weights='distance',
    metric="manhattan"
)
knn.fit(X_train_knn, y_train)


Final test evaluation

In [ ]:
# get the probability that each borrower belongs to class 1 (DEFAULT)
# [:, 1] selects ONlY the possibility of class 1 for each borrower, which is what I want to use for thresholding.

probabilities = knn.predict_proba(
    X_test_knn)[:, 1] 

# if the probability of DEFAULT is greater than or equal to the threshold,
# predict 1. Otherwise predict 0. Doing this prioritizes recall by utilizing our proven best threshold.
y_pred = (probabilities >= best_threshold).astype(int) 

print("Best k:", best_k)
print("Best threshold:", best_threshold)
print(f"\nRecall: {recall_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"F1: {f1_score(y_test, y_pred):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

Overall results are not bad, all of this at the cost of a 20 second runtime.

To get recall to approach .80 without completely sacrificing everything else, I had to filter by only accepting results with >.50 precision.

Even still, F1 is .64 which 

Plot Precision-Recall Curve

In [ ]:
PrecisionRecallDisplay.from_estimator(knn, X_test_knn, y_test)
plt.title("Precision-Recall Curve (KNN)")
plt.show()

Looking at Precision, Recall and F1: Recall and accuracy are good, precision is mediocre, and F1 score is passable

For this kind of dataset, Recall is much more important than Precision

Think about context: If we approve a loan for a customer who will default (false negative), we can lose a lot of money

If we disprove a loan for a customer who won't default (false negative), we don't gain money, but we also don't lose money

In this case false positives are not as damaging as false negatives. So we really need to minimize FNs.

With all of that considered, these numbers still aren't great. The precision, recall and F1 score should all be much higher.

I want to aim for >.80 for recall and accuracy and >.60 for precision, at least.

KNN isn't a good fit for this kind of dataset (too large, too many features)




# 5. Decision Trees

I decided upon Decision Trees because from my knowlege, Decision Trees are well suited for larger datasets. The thing I have to watch out for is overfitting, which is a common issue with DT's.

I should also clarify that I'm not concerned about underfitting at all, because:

1. I have a lot of samples in the training set

2. The dataset has a lot of features. When I experimented with dropping some features, my scores would go down drastically, so that's why I avoided it


**Starting from scratch from KNN.**

In [ ]:
X_dt = df.drop([
    'customer_id', 
    'Current_loan_status'
    #,'customer_age'
    #, 'home_ownership_OTHER'
    ], 
    axis = 1
    )
y = df['Current_loan_status'] 
X_train_dt, X_test_dt, y_train, y_test = train_test_split( X_dt, y, random_state = 42, test_size = 0.2, shuffle = True)

**Decision Tree #1, GridSearchCV:** Automating decision tree hyperparameter combinations first to see how that works

In [ ]:
decisionTree=DecisionTreeClassifier(random_state = 42)

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10, 20, 50],
    'min_samples_leaf': [1, 2, 4, 8, 12, 20],
    'max_features': [None],
    'class_weight': [None, 'balanced'],
    'ccp_alpha': [0.0, 0.000001, 0.000005, 0.00001, 0.00005]
}

dt_recall_search = GridSearchCV(
    estimator=decisionTree, 
    param_grid=param_grid, 
    cv=5, 
    scoring='recall', 
    n_jobs=-1, 
    verbose=1
)

dt_recall_search.fit(X_train_dt, y_train)

print(f"Best Parameters:\n {dt_recall_search.best_params_}")
print(f"Best Cross-Validation Score: {dt_recall_search.best_score_:.4f}")

# Only and final decision tree model
best_model = dt_recall_search.best_estimator_

# Predict once using the exact model chosen by your existing GridSearchCV
y_pred_dt = best_model.predict(X_test_dt)

Doing the first gridsearch and testing all these parameters takes 5 minutes to do

In [ ]:
print(f"Training accuracy: {best_model.score(X_train_dt, y_train):.4f}")
print(f"Test accuracy: {best_model.score(X_test_dt, y_test):.4f}")
print("Tree depth:", best_model.get_depth())
print("Number of leaves:", best_model.get_n_leaves())

print(f"\nRecall: {recall_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"F1: {f1_score(y_test, y_pred_dt):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(
    "Confusion matrix:\n",
    confusion_matrix(y_test, y_pred_dt)
)

Optimizing this model for recall gives it .79

This is only +1 difference between KNN recall, but the difference is:

KNN Precision: .54 -> .63 (.09)

KNN F1: .64 -> .70 (.06)

KNN Accuracy: .81 -> .86 (.05)

I improved on every metric, at the cost of a 5 minute run time, wheras KNN took 

Since this Decision Tree + GridSearch worked so well, I am going to try another one, this time priortizing f1

I'm almost going to narrow my list of parameter options now that I know which metrics are ideal, to make it run faster.



Feature Importance

In [ ]:
importance = pd.Series(
    best_model.feature_importances_,
    index=X_train_dt.columns
)

print(importance.sort_values(ascending=False))

Precision-Recall Curve 

In [ ]:
PrecisionRecallDisplay.from_estimator(best_model, X_test_dt, y_test)
plt.title("Precision-Recall Curve (Decision Tree #1, GridSearchCV)")
plt.show()

**Decision Tree #2:** GridSearchCV Decision Tree that optimizes for F1-score over Recall. With this model, I'm hope to achieve a more balanced output.

In [ ]:
X_dt = df.drop([
    'customer_id', 
    'Current_loan_status'
    #,'customer_age'
    #, 'home_ownership_OTHER'
    ], 
    axis = 1
    )
y = df['Current_loan_status'] 
X_train_dt, X_test_dt, y_train, y_test = train_test_split( X_dt, y, random_state = 42, test_size = 0.2, shuffle = True)

In [ ]:
decisionTree=DecisionTreeClassifier(random_state = 42)

# param_grid = {
#     'criterion': ['gini', 'entropy'],
#     'max_depth': [10, 15, 20],
#     'min_samples_split': [2, 5, 10, 20, 50],
#     'min_samples_leaf': [1, 2, 4, 8, 12],
#     'max_features': [None],
#     'class_weight': ['balanced'],
#     'ccp_alpha': [0.0, 0.000001]
# }

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5, 10, 20, 50],
    'min_samples_leaf': [1, 2, 4, 8, 12, 20],
    'max_features': [None],
    'class_weight': [None, 'balanced'],
    'ccp_alpha': [0.0, 0.000001, 0.000005, 0.00001, 0.00005]
}

dt_f1_search = GridSearchCV(
    estimator=decisionTree, 
    param_grid=param_grid, 
    cv=5, 
    scoring='f1', 
    n_jobs=-1, 
    verbose=1
)

dt_f1_search.fit(X_train_dt, y_train)

print(f"Best Parameters:\n {dt_f1_search.best_params_}")
print(f"Best Cross-Validation Score: {dt_f1_search.best_score_:.4f}")

# Only and final decision tree model
best_model = dt_f1_search.best_estimator_

# Predict once using the exact model chosen by your existing GridSearchCV
y_pred_dt = best_model.predict(X_test_dt)

In [ ]:
print(f"Training accuracy: {best_model.score(X_train_dt, y_train):.4f}")
print(f"Test accuracy: {best_model.score(X_test_dt, y_test):.4f}")
print("Tree depth:", best_model.get_depth())
print("Number of leaves:", best_model.get_n_leaves())

print(f"\nRecall: {recall_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"F1: {f1_score(y_test, y_pred_dt):.4f}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")
print(
    "Confusion matrix:\n",
    confusion_matrix(y_test, y_pred_dt)
)

Training + test accuracy both excellent

Recall = 0.65 is subpar, but precision is excellent. If we were priortizing precision, this would be the best model by far.

Of course, we are assuming (for the sake of simplicity) that false negatives are more costly than false positives in this context, but that's an assumption that could change based on what the real-world statistics say.

F1 and Accuracy are also good.

In [ ]:
# Without setting max_depth equal to 10, I noticed that training would = 1. 
# This indicates overfitting, so setting a max_Depth prevented this.
decisionTree = DecisionTreeClassifier(
    max_depth = 9, 
    random_state = 42,
    class_weight = "balanced",
    criterion = "gini",
    min_samples_split = 2,
    min_samples_leaf = 1
            )

decisionTree.fit(X_train_dt, y_train)

path = decisionTree.cost_complexity_pruning_path(X_train_dt, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

training_score = decisionTree.score(X_train_dt, y_train)
test_score = decisionTree.score(X_test_dt, y_test)

print( f'Training accuracy: {training_score:.4f}' )
print( f'Test accuracy: {test_score:.4f}' )

y_pred = decisionTree.predict(X_test_dt)

print("\nRecall:", f'{recall_score(y_test, y_pred):.4f}')
print("Precision:", f'{precision_score(y_test, y_pred):.4f}')
print("F1", f'{f1_score(y_test, y_pred):.4f}')
print("Accuracy:", f'{accuracy_score(y_test, y_pred):.4f}')
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
PrecisionRecallDisplay.from_estimator(decisionTree, X_test_dt, y_test)
plt.title("Precision-Recall Curve (Decision Tree #2, Manually Selected)")
plt.show()

**Comparing Decision Tree Results:** 

DT #1:

Recall = .7914

Precision = 0.6354

F1 = 0.7049

Accuracy = 0.8601

DT #2:

Recall = .7544 (down .04)

Precision = .7223 (up .09)

F1 = 0.7380 (up .03)

Accuracy = .8869 (up .02)

Training and testing scores of both are both good. GridSearchCV DT has better recall by .04, but every other metric suffer significantly. #2 is a good balance.

It's good to optimize recall, but it's also horrible to have >.90 recall with <.20 precision.

For a real lender, we would have to compare the financial cost of false negatives against false positives rather than choosing solely from these three metrics. 



### UNDER PROGRESS
### FOR RANDOM FOREST - USE RANDOMIZEDSEARCHCV OVER GRID SEARCH?

# 6. Random Forest Ensemble

For my final 2 models, I will try Random Forest Ensemble.

A random forest is an ensemble of many decision trees that aggregate individual predictions to form a consensus. It utilies bagging to train each tree on a random subset of the data, ensuring diversity. 

Similarly to what I did with Decision Trees, I would like to try two versions of it.

**RF Model #1: Modest**

In [ ]:
X_rf = df.drop(
    [
        'customer_id',
        'Current_loan_status',
        #'home_ownership_OTHER'
        #'customer_age'
        #'historical_default'aa
    ],
    axis=1
)

y = df['Current_loan_status']

X_train_rf, X_test_rf, y_train, y_test = train_test_split(
    X_rf,
    y,
    random_state=42,
    test_size=0.2,
    shuffle=True
)

In [ ]:
rf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
    , max_depth = 10,
    n_estimators = 200,
    min_samples_leaf = 1
)

rf.fit(X_train_rf, y_train)

y_pred = rf.predict(X_test_rf)

In [ ]:
training_score = rf.score(X_train_dt, y_train)
test_score = rf.score(X_test_dt, y_test)

print( f'Train: {training_score:.4f}' )
print( f'Test: {test_score:.4f}' )

print("\nRecall:", f'{recall_score(y_test, y_pred):.4f}')
print("Precision:", f'{precision_score(y_test, y_pred):.4f}')
print("F1:", f'{f1_score(y_test, y_pred):.4f}')
print("Accuracy:", f'{accuracy_score(y_test, y_pred):.4f}')
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

So right off the bat I have .78 recall and .67 precision. Compare this to 1st DT where I had .79 recall and .63 precision, I like these results more (and it did not take nearly as long, only a couple seconds, which is a big plus)

In [ ]:
PrecisionRecallDisplay.from_estimator(rf, X_test_rf, y_test)
plt.title("Precision-Recall Curve (Random Forest #1, Modest)")
plt.show()

**RF Model #2: Hyperoptimized for recall**

In [ ]:
rf = RandomForestClassifier(
    random_state=42,
    class_weight='balanced'
    , max_depth = 10,
    n_estimators = 200,
    min_samples_leaf = 1
)

rf.fit(X_train_rf, y_train)

probabilities = rf.predict_proba(X_test_rf)[:, 1] 
results = []

thresholds = [
    0.50, 0.45, 0.41, 0.40, .37, .36, 0.35, 0.30,
    0.25, 0.24, 0.23, 0.22, 0.21,
    0.20, 0.15, 0.10
]

for threshold in thresholds:
    # Predict DEFAULT when its estimated probability reaches the threshold
    y_pred_threshold = (
        probabilities >= threshold
    ).astype(int)

    precision = precision_score(
        y_test,
        y_pred_threshold,
        zero_division=0
    )
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    results.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1
    })

# Keep only results with precision of at least 0.50
acceptable_results = [
    result for result in results
    if result['precision'] >= 0.50
]

if not acceptable_results:
    raise ValueError(
        "No tested threshold produced precision of at least 0.50."
    )

# Among acceptable results, select the one with the highest recall
best_result = max(
    acceptable_results,
    key=lambda result: result['recall']
)

best_threshold = best_result['threshold']
print("Best result:", best_result)

In [ ]:
rf = RandomForestClassifier(  # run random forest again with other threshold
    random_state=42,
    class_weight='balanced'
    , max_depth = 10,
    n_estimators = 200,
    min_samples_leaf = 1
)

rf.fit(X_train_rf, y_train)

probabilities = rf.predict_proba(X_test_rf)[:, 1] 
y_pred = (probabilities >= best_threshold).astype(int)

In [ ]:
print( f'Train: {training_score:.4f}' )
print( f'Test: {test_score:.4f}' )


print("\nRecall:", f'{recall_score(y_test, y_pred):.4f}')
print("Precision:", f'{precision_score(y_test, y_pred):.4f}')
print("F1:", f'{f1_score(y_test, y_pred):.4f}')
print("Accuracy:", f'{accuracy_score(y_test, y_pred):.4f}')
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

In [ ]:
PrecisionRecallDisplay.from_estimator(rf, X_test_rf, y_test)
plt.title("Precision-Recall Curve (Random Forest #2, Hyperoptimized )")
plt.show()

In [ ]:
importance = pd.Series(
    rf.feature_importances_,
    index=X_train_rf.columns
)

print(importance.sort_values(ascending=False))

# 7. Final Conclusions


The findings of my project demonstrate that the two Random Forest models beat out the other models handily. They avoid overfitting, process in a short amount of time, and yield the best results, boasting the best overall scores. The Precision-Recall curves of the two models also reinforce this, with an AP of 0.84 (while KNN = 0.77 and DT = 0.82). 

Out of the two models, the 1st Modest RF has the more balanced output, with recall at .78, precision at .67 and F1 at .72. The 2nd model is more biased for recall, at .85, while precision is .53, and F1 .65. Assuming that False Negatives are more costly for the hypothetical loaning firm than False Positives are (fair assumption), the 2nd Hyperoptimized model is my pick for the best model of this project.